# YouTube Transcript Fetcher — KyivNotKiev

Fetches English transcripts for all YouTube videos in the KyivNotKiev dataset.
Downloads video IDs from HuggingFace, fetches transcripts, saves to parquet.

Run each cell in order. Progress is logged after every single video.
Checkpoints every 100 videos — safe to restart if Colab disconnects.

In [ ]:
!pip install -q youtube-transcript-api huggingface_hub pandas pyarrow

In [ ]:
# Load video IDs from HuggingFace (public, no auth needed)
from huggingface_hub import hf_hub_download
import pandas as pd

path = hf_hub_download(
    repo_id="KyivNotKiev/toponym-adoption-data",
    filename="data/raw_youtube.parquet",
    repo_type="dataset",
)
yt = pd.read_parquet(path)
vids = yt[["video_id", "pair_id"]].drop_duplicates(subset=["video_id"])
vid_to_pair = dict(zip(vids["video_id"], vids["pair_id"]))
print(f"Loaded {len(vids):,} unique video IDs")

In [ ]:
import json, os, time

CHECKPOINT_PATH = "yt_checkpoint.json"
OUT_PATH = "youtube_transcripts.parquet"
BATCH_SIZE = 100  # checkpoint every N videos
DELAY = 1.0       # seconds between requests

# Resume from checkpoint
done = set()
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        done = set(json.load(f))

results = []
if os.path.exists(OUT_PATH):
    prev = pd.read_parquet(OUT_PATH)
    results = prev.to_dict("records")

remaining = [v for v in vids["video_id"].tolist() if v not in done]
print(f"Done: {len(done):,} | Transcripts so far: {len(results):,} | Remaining: {len(remaining):,}")

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from datetime import datetime

ytt = YouTubeTranscriptApi()
no_subs = 0
errors = 0
t0 = time.time()

for i, vid in enumerate(remaining):
    try:
        t = ytt.fetch(vid, languages=["en"])
        text = " ".join(s.text for s in t)
        if len(text) > 20:
            results.append({
                "video_id": vid,
                "pair_id": int(vid_to_pair.get(vid, -1)),
                "transcript": text[:10000],
                "transcript_len": len(text),
            })
            status = f"✓ {len(text):,} chars"
        else:
            no_subs += 1
            status = "· no subs"
    except Exception as e:
        ename = type(e).__name__
        if "IpBlocked" in ename or "RequestBlocked" in ename:
            print(f"\n⚠️  IP BLOCKED at video {i+1}. Wait 1 hour and rerun this cell.")
            print(f"Progress saved: {len(results):,} transcripts")
            # Save before stopping
            done.add(vid)
            with open(CHECKPOINT_PATH, "w") as f:
                json.dump(list(done), f)
            pd.DataFrame(results).to_parquet(OUT_PATH, index=False)
            break
        no_subs += 1
        status = f"✗ {ename}"
        errors += 1

    done.add(vid)

    # Log every video
    elapsed = time.time() - t0
    rate = (i + 1) / elapsed if elapsed > 0 else 0
    pct = (i + 1) / len(remaining) * 100
    eta_min = (len(remaining) - i - 1) / rate / 60 if rate > 0 else 0
    print(f"  [{i+1:,}/{len(remaining):,}] ({pct:.1f}%) {vid} {status} | total={len(results):,} | ETA={eta_min:.0f}min", end="\r")

    # Checkpoint
    if (i + 1) % BATCH_SIZE == 0:
        with open(CHECKPOINT_PATH, "w") as f:
            json.dump(list(done), f)
        pd.DataFrame(results).to_parquet(OUT_PATH, index=False)
        print(f"\n  💾 Checkpoint: {len(results):,} transcripts, {no_subs:,} no subs, {errors:,} errors ({rate:.1f}/sec)")

    time.sleep(DELAY)

else:
    # Loop completed without break
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(list(done), f)
    if results:
        pd.DataFrame(results).to_parquet(OUT_PATH, index=False)
    print(f"\n\n✅ Done: {len(results):,} transcripts, {no_subs:,} no subs, {errors:,} errors")

In [ ]:
# Check results
if os.path.exists(OUT_PATH):
    df = pd.read_parquet(OUT_PATH)
    print(f"Transcripts: {len(df):,}")
    print(f"Pairs: {df['pair_id'].nunique()}")
    print(f"Median length: {df['transcript_len'].median():.0f} chars")
    print(f"Total text: {df['transcript_len'].sum():,} chars")
    print(f"\nSample:")
    for _, r in df.head(3).iterrows():
        print(f"  [{r['pair_id']}] {r['transcript'][:100]}...")
else:
    print("No results yet")

In [ ]:
# Download the result file
# In Colab: click the file icon on the left → find youtube_transcripts.parquet → download
# Or use:
try:
    from google.colab import files
    files.download(OUT_PATH)
    print("Download started")
except:
    print(f"Download manually: {os.path.abspath(OUT_PATH)}")